In [10]:
import qewton.equations as eq

x = eq.Variable("x", dim=2)
y = eq.Variable("y", dim=2)
Omega, Y = eq.Domain("Omega", ambient=x), eq.Domain("Y", ambient=y)

a   = eq.FunctionSymbol(eq.Variable("a",   dim=1), y)  
phi = eq.FunctionSymbol(eq.Variable("phi", dim=2), y)   
K   = eq.FunctionSymbol(eq.Variable("K",   dim=(2, 2)), x)
u   = eq.FunctionSymbol(eq.Variable("u",   dim=1), x)
f   = eq.FunctionSymbol(eq.Variable("f",   dim=1), x)

I2 = eq.Const(((1.0, 0.0), (0.0, 1.0)))
flux = a * (eq.grad(phi, y)*u + I2)

cell  = eq.Eq(-eq.div(flux, y), 0).on(Omega*Y)
Kdef  = eq.Eq(K, eq.mean(flux, Y)).on(Omega)
macro = eq.Eq(-eq.div(eq.dot(K, eq.grad(u, x)), x), f).on(Omega)

problem = eq.Problem((cell, Kdef, macro))
task = problem.task(unknown=[phi, K, u], fixed=[f, a, Omega], varying=[Y])

In [11]:
ctx = eq.run_pipeline(task)
print(eq.report(ctx))

Task
  unknown: K, phi, u
  fixed: a, f
  varying sets: Y
  batch axis of Y: p_Y
Signatures
  K : x × p_Y → ℝ^(2, 2)  [unknown; inherits p_Y]  on Omega
  a : y → ℝ  [fixed]  on Omega × Y, Y
  f : x → ℝ  [fixed]  on Omega
  phi : y × p_Y → ℝ^(2,)  [unknown; inherits p_Y]  on Omega × Y, Y
  u : x × p_Y → ℝ  [unknown; inherits p_Y]  on Omega, Omega × Y, Y
Normalized
  constraints:
    1. −∇_y·(a·(∇_y phi·u + [(1, 0), (0, 1)])) = 0 on Omega × Y
    2. K − ∫_{Y} (a·(∇_y phi·u + [(1, 0), (0, 1)])) / ∫_{Y} 1 = 0 on Omega
    3. −∇_x·dot(K, ∇_x u) − f = 0 on Omega
Batching
  p_Y (Y): geometry; reaches K, phi, u
  instances are independent: parallel over the batch
Classification
  equations:
    −∇_y·(a·(∇_y phi·u + [(1, 0), (0, 1)])) = 0 on Omega × Y: elliptic  (highest time order 0, highest space order 2)
    K − ∫_{Y} (a·(∇_y phi·u + [(1, 0), (0, 1)])) / ∫_{Y} 1 = 0 on Omega: first-order stationary  (highest time order 0, highest space order 1)
    −∇_x·dot(K, ∇_x u) − f = 0 on Omega: ellipt